# 1. Parse with Docling

This notebook parses documents with Docling and writes `doc.json` and `doc.md` outputs to a processed Volume.

Inputs:
- Volume path with raw documents

Outputs:
- `doc.json`
- `doc.md`

In [1]:
%pip install uv
%sh uv pip install .
%restart_python

/Users/scott.mckean/Repos/multimodal-accelerator/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


UsageError: Line magic function `%sh` not found (But cell magic `%%sh` exists, did you mean that instead?).


In [7]:
import mlflow
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import VolumeType

from utils import is_local_env

config = mlflow.models.ModelConfig(development_config="./config.yaml").to_dict()

In [8]:
if is_local_env():
    from databricks.connect import DatabricksSession

    spark = DatabricksSession.builder.serverless().getOrCreate()

This section of code sets up volumes if needed.

In [5]:
CATALOG = config["catalog"]
SCHEMA = config["schema"]
INPUT_VOLUME = config["input_volume"]
OUTPUT_VOLUME = config["output_volume"]

local_input_path = config.get("local_input_path")
local_output_path = config.get("local_output_path")

client = WorkspaceClient()

try:
    client.catalogs.create(name=CATALOG)
except Exception as e:
    print(e)
    pass

try:
    client.schemas.create(catalog_name=CATALOG, name=SCHEMA)
except Exception as e:
    print(e)
    pass

try:
    client.volumes.create(
        catalog_name=CATALOG,
        schema_name=SCHEMA,
        name=OUTPUT_VOLUME,
        volume_type=VolumeType.MANAGED,
    )
except Exception:
    pass

print("Setup complete.")

Catalog 'main' already exists
Schema 'default' already exists
Setup complete.


In [ ]:
# Optional: copy example docs from the repo into the input directory.
# Uncomment to use the sample files under examples/.
# from utils import copy_local_inputs, resolve_input_root
# input_root = resolve_input_root(CATALOG, SCHEMA, INPUT_VOLUME, local_input_path)
# copied = copy_local_inputs("examples", input_root)
# print(f"Copied {copied} example files to {input_root}")

In [6]:
from pathlib import Path

from docling.document_converter import DocumentConverter

from utils import (
    get_file_list,
    print_processing_summary,
    resolve_input_root,
    resolve_output_root,
    sanitize_filename,
)

input_root = resolve_input_root(CATALOG, SCHEMA, INPUT_VOLUME, local_input_path)
output_root = resolve_output_root(CATALOG, SCHEMA, OUTPUT_VOLUME, local_output_path)

input_files = get_file_list(input_root, pattern="*.*")

converter = DocumentConverter()
results = []

for file_path in input_files:
    safe_stem = sanitize_filename(file_path.stem)
    output_dir = output_root / safe_stem
    output_dir.mkdir(parents=True, exist_ok=True)

    try:
        result = converter.convert(str(file_path))
        document = result.document

        document.save_as_json(output_dir / "doc.json")
        document.save_as_markdown(output_dir / "doc.md")

        results.append(
            {
                "status": "success",
                "input_path": str(file_path),
                "output_path": str(output_dir),
            }
        )
    except Exception as exc:
        results.append(
            {
                "status": "error",
                "input_path": str(file_path),
                "error": str(exc),
            }
        )

print_processing_summary(results, "docling_parse")


DOCLING_PARSE PROCESSING SUMMARY
Total files: 6
Successful: 6
Failed: 0
Success rate: 100.0%
